In [1]:
# Added for the public reproduction package: load the pseudonymized CSV instead of the
# private spreadsheet and write any output files to ../output/notebooks/. See repro_data.py.
import repro_data


In [2]:
"""
Correlación de Spearman entre SUS UX y SUS de seguridad, por grupo de tratamiento,
con gráficos de dispersión (combinado + PDF individual por grupo y global).

Exclusiones aplicadas sobre los 67 casos originales:
  1) Filtro habitual (n=60): se descartan los respondientes que no contestaron
     NINGÚN ítem del bloque de seguridad (7 casos).
  2) Exclusión puntual: participante de la fila 54 de Excel (índice 52 en
     pandas, 0-based), que no respondió la PRIMERA pregunta de seguridad.
     Aunque su "seguridad sus" tiene un valor calculado, ese ítem faltante
     invalida el puntaje para este análisis -> se excluye también.

Resultado esperado: n=59 casos para el análisis de correlación.
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

# Coloca el archivo en la misma carpeta que este script, o reemplazá por la ruta completa
ARCHIVO_EXCEL = "Wallet-pattern-.xlsx"
HOJA = "Datos cuanti"
COL_GRUPO = "Grupo"
COL_SUS_UX = "SUS UX"
COL_SUS_SEG = "seguridad sus"

CARPETA_SALIDA = "resultados_correlacion"

# Fila de Excel a excluir (1-based, contando el encabezado como fila 1)
EXCEL_ROW_TO_EXCLUDE = 54
# Conversión a índice de pandas: fila 1 = encabezado, fila 2 = índice 0
PANDAS_IDX_TO_EXCLUDE = EXCEL_ROW_TO_EXCLUDE - 2

# Los 6 ítems del bloque de seguridad, usados para el filtro habitual (n=60)
SEC_ITEM_COLS = [
    "Siento que este método de envío de criptoactivos es lo suficientemente "
    "seguro para valores que representen hasta 1 (un) ingreso mensual",
    "Siento que este método de envío de criptoactivos es no lo suficientemente "
    "seguro incluso para valores que representen hasta 2 (dos) ingresos mensuales",
    "Siento que este método de envío de criptoactivos es lo suficientemente "
    "seguro para la totalidad del monto de todos mis criptoactivos con este método.",
    "Siento que este método de envío de criptoactivos no es lo suficientemente "
    "seguro para todos mis criptoactivos con este método.",
    "Siento que este método de envío de criptoactivos es lo suficientemente "
    "seguro independiente del monto que representan.",
    "Siento que este método de envío de criptoactivos no es lo suficientemente "
    "seguro independiente del monto que representan.",
]


def graficar_grupo(sub, grupo, rho, p_valor, ax=None, standalone_path=None, titulo=None):
    """Dibuja un scatter SUS UX vs seguridad para un grupo (o el global).

    Si ax se pasa, dibuja sobre ese eje (para el grid combinado).
    Si no, crea una figura nueva; con standalone_path la guarda en PDF.
    """
    standalone = ax is None
    if standalone:
        fig_ind, ax = plt.subplots(figsize=(5, 4))

    ax.scatter(sub[COL_SUS_UX], sub[COL_SUS_SEG], alpha=0.7, edgecolor="black")
    ax.set_xlabel("SUS UX")
    ax.set_ylabel("Perceived Security SUS")
    etiqueta = titulo if titulo else f"Group {grupo} (n={len(sub)})"
    ax.set_title(etiqueta, fontsize=10)

    # --- rho y p como texto dentro del gráfico (no en el título) ---
    ax.text(
        0.05, 0.95,
        f"rho = {rho:.3f}\np = {p_valor:.3f}",
        transform=ax.transAxes,
        fontsize=9,
        verticalalignment="top",
        horizontalalignment="left",
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="gray", alpha=0.85),
    )

    if standalone:
        fig_ind.tight_layout()
        if standalone_path:
            fig_ind.savefig(standalone_path, bbox_inches="tight")
        plt.close(fig_ind)


def main():
    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    df = repro_data.read_datos_cuanti()

    # --- Verificación de la fila a excluir ---
    fila_excluida = df.loc[PANDAS_IDX_TO_EXCLUDE]
    assert pd.isna(fila_excluida[SEC_ITEM_COLS[0]]), (
        f"La fila en índice {PANDAS_IDX_TO_EXCLUDE} (Excel {EXCEL_ROW_TO_EXCLUDE}) "
        "no tiene NaN en el primer ítem de seguridad como se esperaba. Revisar el mapeo de fila."
    )
    print(
        f"Excluyendo participante — fila Excel {EXCEL_ROW_TO_EXCLUDE} "
        f"(índice pandas {PANDAS_IDX_TO_EXCLUDE}), grupo {int(fila_excluida[COL_GRUPO])}, "
        "no respondió la primera pregunta de seguridad."
    )

    # --- Filtro 1: descartar quienes no respondieron ningún ítem de seguridad ---
    respondio_algo_seguridad = df[SEC_ITEM_COLS].notna().any(axis=1)
    df_filtrado = df[respondio_algo_seguridad].copy()
    print(f"Tras filtro habitual (respondió algo de seguridad): n={len(df_filtrado)}")

    # --- Filtro 2: excluir puntualmente la fila 54 de Excel ---
    df_filtrado = df_filtrado.drop(index=PANDAS_IDX_TO_EXCLUDE)
    print(f"Tras excluir fila Excel {EXCEL_ROW_TO_EXCLUDE}: n={len(df_filtrado)}")

    grupos = sorted(df_filtrado[COL_GRUPO].dropna().unique())
    ncols = 2
    nrows = -(-len(grupos) // ncols)  # techo (ceil)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    resultados = []
    for ax, grupo in zip(axes, grupos):
        sub = df_filtrado[df_filtrado[COL_GRUPO] == grupo].dropna(subset=[COL_SUS_UX, COL_SUS_SEG])
        rho, p_valor = spearmanr(sub[COL_SUS_UX], sub[COL_SUS_SEG])
        resultados.append({"Grupo": grupo, "n": len(sub), "rho_spearman": rho, "p_valor": p_valor})
        graficar_grupo(sub, grupo, rho, p_valor, ax=ax)
        # --- PDF individual por grupo ---
        graficar_grupo(
            sub, grupo, rho, p_valor,
            standalone_path=os.path.join(CARPETA_SALIDA, f"spearman_grupo_{grupo}.pdf"),
        )

    for ax in axes[len(grupos):]:
        ax.set_visible(False)

    fig.suptitle("Spearman Correlation: SUS UX vs. Perceived Security by Group", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    ruta_combinado = os.path.join(CARPETA_SALIDA, "spearman_todos_los_grupos.pdf")
    fig.savefig(ruta_combinado, bbox_inches="tight")
    print(f"\nGráfico combinado guardado en: {ruta_combinado}")
    plt.close(fig)

    # --- Correlación global ---
    rho_g, p_g = spearmanr(df_filtrado[COL_SUS_UX], df_filtrado[COL_SUS_SEG])
    resultados.append({"Grupo": "Global", "n": len(df_filtrado), "rho_spearman": rho_g, "p_valor": p_g})

    # --- PDF individual de la correlación global (todos los grupos juntos) ---
    graficar_grupo(
        df_filtrado, "Global", rho_g, p_g,
        standalone_path=os.path.join(CARPETA_SALIDA, "spearman_global.pdf"),
        titulo=f"All Groups Combined (n={len(df_filtrado)})",
    )

    resumen = pd.DataFrame(resultados)
    print("\nResumen de correlaciones (subconjunto filtrado):")
    print(resumen.to_string(index=False))
    ruta_resumen = os.path.join(CARPETA_SALIDA, "resumen_correlacion_spearman.xlsx")
    resumen.to_excel(ruta_resumen, index=False)
    print(f"\nResumen exportado a: {ruta_resumen}")

    ruta_datos = os.path.join(CARPETA_SALIDA, "datos_filtrados.xlsx")
    df_filtrado.to_excel(ruta_datos, index=False)
    print(f"Datos filtrados exportados a: {ruta_datos}")


if __name__ == "__main__":
    main()

Excluyendo participante — fila Excel 54 (índice pandas 52), grupo 2, no respondió la primera pregunta de seguridad.
Tras filtro habitual (respondió algo de seguridad): n=60
Tras excluir fila Excel 54: n=59



Gráfico combinado guardado en: resultados_correlacion/spearman_todos_los_grupos.pdf

Resumen de correlaciones (subconjunto filtrado):
 Grupo  n  rho_spearman  p_valor
     1 14      0.606516 0.021472
     2 13      0.817301 0.000646
     3 18      0.534867 0.022191
     4 14     -0.607581 0.021182
Global 59      0.460467 0.000243

Resumen exportado a: resultados_correlacion/resumen_correlacion_spearman.xlsx
Datos filtrados exportados a: resultados_correlacion/datos_filtrados.xlsx
